# 12 — HyperTempNet: analisi del modello + explainability

Analizza il modello **novel** (HyperTempNet) partendo dal **checkpoint salvato** (nessun
riaddestramento). Tre parti:
1. **Summary** dei layer e conteggio parametri;
2. **Walkthrough delle shape** stadio per stadio (raw → segmenti → ipergrafo → logits);
3. **Explainability**: (a) l'incidenza degli iperarchi `H`, (b) integrated gradients sull'input.

> Env `daniele_311`. Serve `results/checkpoints/hypertempnet_best.pt` (creato da
> `train_save_extract`) e, per la parte IG, `results/ig_hypertempnet.npz`.
> Le sezioni 3–4 usano i file precomputati quando ci sono, altrimenti ricalcolano dal checkpoint.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, sys; sys.path.insert(0, os.path.abspath('.'))
import numpy as np, torch, matplotlib.pyplot as plt
import track3_config as C, track3_preproc as P, track3_train as T, track3_infer as I

model, meta = I.load_checkpoint()
DEVICE = meta['device']
print('checkpoint:', meta['model_name'], '| protocollo', meta['protocol'], '| test_bacc', round(meta['test_bacc'],4))
print('config modello:', meta['model_kwargs'])
print('classi:', meta['class_names'])

## §1 — Summary: layer e parametri
Prova `torchinfo`; in fallback una tabella manuale per modulo di primo livello.

In [ ]:
print(model)
n_par = sum(p.numel() for p in model.parameters())
n_tr  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nParametri totali: {n_par:,}  (allenabili: {n_tr:,})\n')

try:
    from torchinfo import summary
    summary(model, input_size=(1, C.N_CHANNELS, meta['n_times']), device=DEVICE,
            col_names=('output_size', 'num_params'), depth=2)
except Exception as e:
    print(f'[torchinfo non disponibile: {e}] -> tabella manuale per modulo:')
    print(f"{'modulo':22s}{'parametri':>12s}")
    for name, mod in model.named_children():
        p = sum(q.numel() for q in mod.parameters())
        print(f'{name:22s}{p:>12,}')
    # E = iperarchi appresi (Parameter, non modulo)
    print(f"{'E (iperarchi)':22s}{model.E.numel():>12,}   shape {tuple(model.E.shape)}")

## §2 — Walkthrough delle shape (dal raw ai logits)
Un trial reale attraversa il modello; stampiamo la shape a ogni stadio per capire cosa fa.

In [ ]:
# un trial reale (soggetto 1, primo test)
d = P.preprocess_subject(1, merge_val_into_train=False, resample_to=None, standardize=True, **C.PP_MINIMAL)
data, _ = T._prepare_inputs(d, 'raw')
x = torch.tensor(data['X_test'][:1], dtype=torch.float32, device=DEVICE)  # (1,64,795)
model.eval()
with torch.no_grad():
    print(f'{"input raw EEG":34s}{tuple(x.shape)}')
    h = torch.cat([b(x.unsqueeze(1)) for b in model.branches], 1)
    print(f'{"conv multi-scala (4 scale x F)":34s}{tuple(h.shape)}  <- (B, Fc, canali, tempo)')
    h = model.spatial(h).squeeze(2)
    print(f'{"conv spaziale (collassa canali)":34s}{tuple(h.shape)}  <- (B, Fc, tempo)')
    B, Fc, Tt = h.shape; seg = Tt // model.K
    h = h[:, :, :seg*model.K].reshape(B, Fc, model.K, seg).mean(3)
    print(f'{"segmentazione temporale":34s}{tuple(h.shape)}  <- K={model.K} segmenti (nodi)')
    feat = model.proj(h.transpose(1, 2))
    print(f'{"proiezione nodi":34s}{tuple(feat.shape)}  <- (B, K nodi, hidden)')
    H = torch.softmax(feat @ model.E.T / (model.hidden**0.5), dim=2)
    print(f'{"incidenza soft H":34s}{tuple(H.shape)}  <- (B, K nodi, n_edges iperarchi)')
    o = torch.relu(model.hg1(feat, H)); o = torch.relu(model.hg2(o, H)); pooled = o.mean(1)
    print(f'{"2x HGNN + mean pool":34s}{tuple(pooled.shape)}  <- (B, hidden)')
    logits = model.clf(pooled)
    print(f'{"classificatore":34s}{tuple(logits.shape)}  <- (B, n_classi)')
    print('\npredizione:', meta['class_names'][int(logits.argmax(1))], '| vera:', meta['class_names'][int(data["y_test"][0])])

## §3 — Explainability (a): l'incidenza degli iperarchi `H` è interpretabile?
Idea: se gli iperarchi imparano *motivi discriminanti*, `H` dovrebbe essere **netta** (pochi edge
dominano) e **diversa per parola**. Misuriamo l'entropia di `H` e guardiamo `H` medio per classe.

In [ ]:
# H su un campione reale di test (dal checkpoint) -> entropia
def _test_X(s):
    dd = P.preprocess_subject(s, merge_val_into_train=False, resample_to=None, standardize=True, **C.PP_MINIMAL)
    return T._prepare_inputs(dd, 'raw')[0]['X_test']
Xs = np.concatenate([_test_X(s) for s in [1, 2, 3]], 0)
H = I.incidence(model, Xs)                       # (N,12,8)
ent = -(H*np.log(H+1e-12)).sum(2) / np.log(H.shape[2])   # entropia normalizzata per segmento
print(f'entropia normalizzata di H: media={ent.mean():.3f}  (1.0 = uniforme piatta, 0 = one-hot)')
print(f'peso max di un iperarco:    {H.max():.3f}          (uniforme = {1/H.shape[2]:.3f})')
print('=> H e quasi UNIFORME: gli iperarchi NON selezionano motivi distinti.')

# H medio per classe (usa il file precomputato sull intero test se presente)
p = 'results/interp_hypertempnet.npz'
if os.path.exists(p):
    dd = np.load(p, allow_pickle=True); meanH = dd['meanH']; cls = [str(c) for c in dd['class_names']]
    fig, axes = plt.subplots(1, len(cls), figsize=(13, 2.6), constrained_layout=True)
    vmin, vmax = meanH.min(), meanH.max()
    for i, ax in enumerate(axes):
        im = ax.imshow(meanH[i], aspect='auto', cmap='magma', vmin=vmin, vmax=vmax)
        ax.set_title(cls[i], fontsize=10); ax.set_xlabel('iperarco'); ax.set_yticks([])
    axes[0].set_ylabel('segmento (tempo)')
    fig.colorbar(im, ax=axes, shrink=0.8, label='H medio')
    fig.suptitle('Incidenza H media per parola — quasi identica e piatta (range %.3f–%.3f)' % (vmin, vmax))
    plt.savefig(C.FIG_DIR/'hypertempnet_H_per_class.png', dpi=130, bbox_inches='tight'); plt.show()
else:
    print('(manca results/interp_hypertempnet.npz per la heatmap per-classe)')

**Conclusione §3**: `H` è quasi uniforme (entropia ~1.0) e quasi identica tra le parole. Gli iperarchi
**non** imparano motivi temporali interpretabili: il ramo ipergrafo, con `H` uniforme, agisce come un
**pooling temporale morbido e regolarizzato**. Il guadagno dell'ablation (ON vs OFF) è reale, ma il
meccanismo non è "selezione di prototipi". Va scritto così, onestamente.

## §4 — Explainability (b): integrated gradients sull'input
Attribuzione fedele: *quali canali/tempi* il modello usa per ogni parola. Carica
`results/ig_hypertempnet.npz` (prodotto da `build_explainability_artifact` / script IG).

In [ ]:
p = 'results/ig_hypertempnet.npz'
assert os.path.exists(p), 'manca results/ig_hypertempnet.npz — eseguire lo script integrated gradients'
ig = np.load(p, allow_pickle=True)
spatial = ig['spatial']; temporal = ig['temporal']; clab = [str(c) for c in ig['clab']]
cls = [str(c) for c in ig['class_names']]; fs = int(ig['fs']); nT = int(ig['n_times'])
print(f"discriminabilita' saliency  spaziale={float(ig['disc_s']):.3f}  temporale={float(ig['disc_t']):.3f}")

# posizioni 2D dei canali (montaggio standard)
import mne
pos3d = mne.channels.make_standard_montage('standard_1005').get_positions()['ch_pos']
pos2d = np.array([pos3d[c][:2] for c in clab])

# topomap per parola
fig, axes = plt.subplots(1, len(cls), figsize=(13, 3.0), constrained_layout=True)
vmax = spatial.max()
for i, ax in enumerate(axes):
    im, _ = mne.viz.plot_topomap(spatial[i], pos2d, axes=ax, show=False, cmap='viridis',
                                 vlim=(0, vmax), contours=0, sensors=True)
    top = [clab[j] for j in spatial[i].argsort()[::-1][:3]]
    ax.set_title(f"{cls[i]}\n{' · '.join(top)}", fontsize=9)
fig.colorbar(im, ax=axes, shrink=0.7, label='|IG| medio (importanza canale)')
fig.suptitle('Integrated gradients — canali che guidano ogni parola (lateralizzazione)')
plt.savefig(C.FIG_DIR/'hypertempnet_ig_topomaps.png', dpi=130, bbox_inches='tight'); plt.show()

In [ ]:
# profilo temporale della saliency (condiviso tra parole)
t_ms = np.arange(nT)/fs*1000
fig, ax = plt.subplots(figsize=(9, 2.8), constrained_layout=True)
for i in range(len(cls)):
    ax.plot(t_ms, temporal[i], color='teal', alpha=0.25, lw=1)
ax.plot(t_ms, temporal.mean(0), color='teal', lw=2.2, label='media parole')
ax.set_xlabel('tempo nel trial (ms)'); ax.set_ylabel('|IG| medio'); ax.legend(loc='upper left')
ax.set_title('Saliency temporale — quasi identica tra parole (il "quando" non discrimina)')
plt.savefig(C.FIG_DIR/'hypertempnet_ig_temporal.png', dpi=130, bbox_inches='tight'); plt.show()
print('top-4 canali per parola:')
for i, w in enumerate(cls):
    print(f'  {w:9s}: ' + ', '.join(clab[j] for j in spatial[i].argsort()[::-1][:4]))

## Conclusioni
- **Architettura**: **40.549 parametri (0.04M)** — stessa taglia della Shallow, ~100× meno di CBraMod
  (4.0M). Il collo di bottiglia informativo è la **segmentazione temporale** (12 nodi) + l'ipergrafo
  appreso, non la profondità.
- **`H` uniforme (entropia ~1.0)**: gli iperarchi non selezionano motivi interpretabili → il ramo
  ipergrafo è pooling temporale regolarizzato. Onesto, va scritto così.
- **Integrated gradients**: la struttura discriminante è **spaziale** (lateralizzazione per parola,
  disc. 0.12), non temporale (disc. 0.02). Pulito perché ogni parola è pronunciata da tutti i 15
  soggetti → l'identità del soggetto si annulla mediando.

Figure salvate in `results/figures/`. Confronto metriche/protocolli in `RESULTS.md`.